# 7.3 其他对齐方法

本节涵盖RLHF和DPO之外的其他对齐方法：RLAIF、KTO、ORPO、SimPO。这些方法从不同角度简化或改进了对齐流程。

## 方法对比总览

| 方法 | 需要偏好对 | 需要参考模型 | 需要奖励模型 | 核心创新 |
|------|-----------|-------------|-------------|----------|
| RLHF | 是 | 是 | 是 | PPO优化 |
| DPO | 是 | 是 | 否 | 隐式奖励 |
| RLAIF | 是(AI标注) | 是 | 可选 | AI替代人类 |
| KTO | 否(只需好/坏标签) | 是 | 否 | 前景理论 |
| ORPO | 是 | 否 | 否 | 胜率比+SFT合并 |
| SimPO | 是 | 否 | 否 | 平均log概率 |

## 1. RLAIF（RL from AI Feedback）

RLAIF用AI反馈替代人类反馈，大幅降低标注成本，是当前工业界最常用的偏好数据生成方式。

### RLAIF流程
1. 用策略模型采样多个回复
2. 用强模型（如GPT-4）对回复进行偏好排序
3. 用AI标注的偏好数据训练奖励模型或直接用于DPO

### RLAIF的关键设计
- **评分维度**：有用性、安全性、准确性、流畅性等
- **评分策略**：绝对评分 vs 相对比较
- **AI标注提示**：精心设计的提示词决定标注质量
- **一致性验证**：多次标注取多数投票

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)

class AIAnnotator:
    def __init__(self, quality_weight=0.5, safety_weight=0.3, accuracy_weight=0.2):
        self.quality_weight = quality_weight
        self.safety_weight = safety_weight
        self.accuracy_weight = accuracy_weight

    def score_single(self, prompt_emb, response_emb):
        quality = torch.cosine_similarity(prompt_emb, response_emb, dim=-1)
        safety = torch.sigmoid(-response_emb.norm(dim=-1) + 2.0)
        accuracy = torch.sigmoid(response_emb[:, :response_emb.shape[-1]//2].sum(dim=-1) - 1.0)
        total = (self.quality_weight * quality +
                 self.safety_weight * safety +
                 self.accuracy_weight * accuracy)
        return total

    def annotate_pair(self, prompt_emb, response_a_emb, response_b_emb, n_votes=3):
        votes_a = 0
        votes_b = 0
        for _ in range(n_votes):
            noise_a = 0.05 * torch.randn_like(response_a_emb)
            noise_b = 0.05 * torch.randn_like(response_b_emb)
            score_a = self.score_single(prompt_emb, response_a_emb + noise_a)
            score_b = self.score_single(prompt_emb, response_b_emb + noise_b)
            votes_a += (score_a > score_b).float()
            votes_b += (score_b > score_a).float()
        return votes_a / n_votes, votes_b / n_votes

    def batch_annotate(self, prompts, responses_chosen, responses_rejected, n_votes=3):
        scores_chosen = self.score_single(prompts, responses_chosen)
        scores_rejected = self.score_single(prompts, responses_rejected)
        consistency = (scores_chosen > scores_rejected).float().mean()
        return scores_chosen, scores_rejected, consistency

annotator = AIAnnotator(quality_weight=0.5, safety_weight=0.3, accuracy_weight=0.2)

n_pairs = 16
d = 64
prompts = torch.randn(n_pairs, d)
chosen = prompts + 0.3 * torch.randn(n_pairs, d)
rejected = prompts + 0.8 * torch.randn(n_pairs, d)

scores_c, scores_r, consistency = annotator.batch_annotate(prompts, chosen, rejected)

print('=== RLAIF: AI Annotation ===')
print(f'Chosen scores:  mean={scores_c.mean():.3f}, std={scores_c.std():.3f}')
print(f'Rejected scores: mean={scores_r.mean():.3f}, std={scores_r.std():.3f}')
print(f'AI preference consistency: {consistency:.2%}')

vote_a, vote_b = annotator.annotate_pair(prompts[:4], chosen[:4], rejected[:4], n_votes=5)
print(f'\nMulti-vote annotation (5 votes):')
print(f'  Chosen win rate: {vote_a.tolist()}')
print(f'  Rejected win rate: {vote_b.tolist()}')

print(f'\nKey: Multi-vote annotation improves reliability.')
print(f'AI annotation scales to millions of pairs at low cost.')

## 2. KTO（Kahneman-Tversky Optimization）

KTO基于前景理论（Prospect Theory），只需要"好"或"不好"的二元标签，不需要成对偏好数据。这大大降低了数据收集难度。

### 核心思想
- 人类对损失的敏感度高于对收益的敏感度（损失厌恶）
- KTO对负例的惩罚大于对正例的奖励
- 数据获取更简单：只需标注单个回复的好坏

### KTO损失
$$L_{KTO} = \mathbb{E}[\lambda_w \cdot (1 - \sigma(\beta \cdot (z(x,y) - z_{ref})))] + \mathbb{E}[\lambda_l \cdot (1 - \sigma(\beta \cdot (z_{ref} - z(x,y))))]$$

其中 $z(x,y) = \log \pi_\theta(y|x) - \log \pi_{ref}(y|x)$，$\lambda_w > \lambda_l$ 体现损失厌恶

In [ ]:
class KTOTrainer:
    def __init__(self, policy, ref_policy, beta=0.1,
                 desirable_weight=1.0, undesirable_weight=1.0,
                 lr=1e-4, max_grad_norm=1.0):
        self.policy = policy
        self.ref_policy = ref_policy
        self.beta = beta
        self.desirable_weight = desirable_weight
        self.undesirable_weight = undesirable_weight
        self.optimizer = torch.optim.AdamW(policy.parameters(), lr=lr)
        self.max_grad_norm = max_grad_norm
        self.metrics_history = []

    def compute_z_ref(self, dataloader_x, dataloader_labels, n_batches=10):
        all_z = []
        with torch.no_grad():
            for _ in range(n_batches):
                x = next(iter(dataloader_x)) if hasattr(dataloader_x, '__iter__') else dataloader_x
                labels = next(iter(dataloader_labels)) if hasattr(dataloader_labels, '__iter__') else dataloader_labels
                policy_logits = self.policy(x)
                ref_logits = self.ref_policy(x)
                log_pi = F.log_softmax(policy_logits, dim=-1).gather(1, labels.unsqueeze(1)).squeeze(1)
                log_ref = F.log_softmax(ref_logits, dim=-1).gather(1, labels.unsqueeze(1)).squeeze(1)
                all_z.append((log_pi - log_ref))
        return torch.cat(all_z).mean().item()

    def train_step(self, x, labels, is_desirable, z_ref=0.0):
        policy_logits = self.policy(x)
        with torch.no_grad():
            ref_logits = self.ref_policy(x)

        log_pi = F.log_softmax(policy_logits, dim=-1).gather(1, labels.unsqueeze(1)).squeeze(1)
        log_ref = F.log_softmax(ref_logits, dim=-1).gather(1, labels.unsqueeze(1)).squeeze(1)
        z = log_pi - log_ref

        loss_desirable = self.desirable_weight * (1 - torch.sigmoid(self.beta * (z - z_ref)))
        loss_undesirable = self.undesirable_weight * (1 - torch.sigmoid(self.beta * (z_ref - z)))

        losses = torch.where(is_desirable.bool(), loss_desirable, loss_undesirable)
        loss = losses.mean()

        self.optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(self.policy.parameters(), self.max_grad_norm)
        self.optimizer.step()

        with torch.no_grad():
            n_des = is_desirable.sum().item()
            n_undes = (~is_desirable).sum().item()
            des_reward = z[is_desirable.bool()].mean().item() if n_des > 0 else 0
            undes_reward = z[~is_desirable.bool()].mean().item() if n_undes > 0 else 0

        metrics = {
            'loss': loss.item(),
            'desirable_reward': des_reward,
            'undesirable_reward': undes_reward,
            'n_desirable': n_des,
            'n_undesirable': n_undes,
        }
        self.metrics_history.append(metrics)
        return metrics

class SimplePolicy(nn.Module):
    def __init__(self, d=64, v=50):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d, 128), nn.ReLU(), nn.LayerNorm(128), nn.Linear(128, v))
    def forward(self, x):
        return self.net(x)

policy = SimplePolicy()
ref_policy = SimplePolicy()
ref_policy.load_state_dict(policy.state_dict())
for p in ref_policy.parameters():
    p.requires_grad = False

kto_trainer = KTOTrainer(policy, ref_policy, beta=0.1,
                          desirable_weight=1.0, undesirable_weight=1.5)

x = torch.randn(32, 64)
labels = torch.randint(0, 50, (32,))
is_desirable = torch.tensor([1,1,0,1,0,0,1,0,1,1,0,1,1,0,1,0,
                              1,0,1,1,0,1,0,0,1,1,0,1,0,1,1,0])

print('=== KTO Training ===')
for step in range(30):
    metrics = kto_trainer.train_step(x, labels, is_desirable)
    if (step + 1) % 10 == 0:
        print(f'Step {step+1:3d}: loss={metrics["loss"]:.4f}, '
              f'des_reward={metrics["desirable_reward"]:.4f}, '
              f'undes_reward={metrics["undesirable_reward"]:.4f}')

print(f'\nKey: KTO only needs binary labels (desirable/undesirable).')
print(f'undesirable_weight > desirable_weight implements loss aversion.')

## 3. ORPO（Odds Ratio Preference Optimization）

ORPO将SFT和对齐合并为一步，使用胜率比（odds ratio）作为偏好信号，无需参考模型。

### 核心思想
- 传统方法：先SFT，再对齐（两步训练）
- ORPO：在SFT损失中加入胜率比惩罚，一步完成
- 胜率比：$\text{OR}(y_w, y_l) = \frac{P(y_w|x)/(1-P(y_w|x))}{P(y_l|x)/(1-P(y_l|x))}$

### ORPO损失
$$L_{ORPO} = L_{SFT} + \lambda \cdot L_{OR}$$
- $L_{SFT}$：标准的交叉熵损失
- $L_{OR}$：基于胜率比的偏好损失
- $\lambda$：偏好损失权重

In [ ]:
class ORPOTrainer:
    def __init__(self, policy, lambda_or=0.1, lr=1e-4, max_grad_norm=1.0):
        self.policy = policy
        self.lambda_or = lambda_or
        self.optimizer = torch.optim.AdamW(policy.parameters(), lr=lr)
        self.max_grad_norm = max_grad_norm
        self.metrics_history = []

    def train_step(self, x, y_chosen, y_rejected):
        logits = self.policy(x)
        log_probs = F.log_softmax(logits, dim=-1)

        chosen_log_prob = log_probs.gather(1, y_chosen.unsqueeze(1)).squeeze(1)
        rejected_log_prob = log_probs.gather(1, y_rejected.unsqueeze(1)).squeeze(1)

        chosen_prob = chosen_log_prob.exp()
        rejected_prob = rejected_log_prob.exp()

        sft_loss = -chosen_log_prob.mean()

        chosen_odds = chosen_prob / (1 - chosen_prob + 1e-8)
        rejected_odds = rejected_prob / (1 - rejected_prob + 1e-8)
        odds_ratio = chosen_odds / (rejected_odds + 1e-8)
        or_loss = -F.logsigmoid(torch.log(odds_ratio + 1e-8)).mean()

        total_loss = sft_loss + self.lambda_or * or_loss

        self.optimizer.zero_grad()
        total_loss.backward()
        nn.utils.clip_grad_norm_(self.policy.parameters(), self.max_grad_norm)
        self.optimizer.step()

        metrics = {
            'total_loss': total_loss.item(),
            'sft_loss': sft_loss.item(),
            'or_loss': or_loss.item(),
            'odds_ratio': odds_ratio.mean().item(),
        }
        self.metrics_history.append(metrics)
        return metrics

policy = SimplePolicy()
orpo_trainer = ORPOTrainer(policy, lambda_or=0.1, lr=1e-3)

x = torch.randn(32, 64)
y_chosen = torch.randint(0, 50, (32,))
y_rejected = torch.randint(0, 50, (32,))

print('=== ORPO Training ===')
for step in range(30):
    metrics = orpo_trainer.train_step(x, y_chosen, y_rejected)
    if (step + 1) % 10 == 0:
        print(f'Step {step+1:3d}: total={metrics["total_loss"]:.4f}, '
              f'sft={metrics["sft_loss"]:.4f}, or={metrics["or_loss"]:.4f}, '
              f'odds_ratio={metrics["odds_ratio"]:.3f}')

print(f'\nKey: ORPO merges SFT + alignment in one step, no reference model needed.')
print(f'lambda_or controls the strength of preference signal.')

## 4. SimPO（Simple Preference Optimization）

SimPO用序列的平均log概率代替参考模型的log概率，完全不需要参考模型，是最简洁的对齐方法。

### 核心思想
- DPO需要参考模型来计算 $\log \pi_{ref}(y|x)$
- SimPO发现：用 $\frac{1}{|y|}\log \pi_\theta(y|x)$ 替代 $\log \frac{\pi_\theta(y|x)}{\pi_{ref}(y|x)}$ 效果相当
- 这消除了参考模型的显存开销

### SimPO损失
$$L_{SimPO} = -\mathbb{E}[\log \sigma(\beta(\hat{r}(x,y_w) - \hat{r}(x,y_l) - \gamma))]$$

其中 $\hat{r}(x,y) = \frac{\beta}{|y|} \log \pi_\theta(y|x) + \gamma$

- $\gamma$：目标奖励差（target reward margin），确保chosen的奖励比rejected高出一个margin

In [ ]:
class SimPOTrainer:
    def __init__(self, policy, beta=0.1, gamma=0.5, lr=1e-4, max_grad_norm=1.0):
        self.policy = policy
        self.beta = beta
        self.gamma = gamma
        self.optimizer = torch.optim.AdamW(policy.parameters(), lr=lr)
        self.max_grad_norm = max_grad_norm
        self.metrics_history = []

    def train_step(self, x, y_chosen, y_rejected, len_chosen=None, len_rejected=None):
        logits = self.policy(x)
        log_probs = F.log_softmax(logits, dim=-1)

        chosen_log_prob = log_probs.gather(1, y_chosen.unsqueeze(1)).squeeze(1)
        rejected_log_prob = log_probs.gather(1, y_rejected.unsqueeze(1)).squeeze(1)

        if len_chosen is not None and len_rejected is not None:
            chosen_avg_log_prob = chosen_log_prob / len_chosen.float()
            rejected_avg_log_prob = rejected_log_prob / len_rejected.float()
        else:
            chosen_avg_log_prob = chosen_log_prob
            rejected_avg_log_prob = rejected_log_prob

        implicit_reward_chosen = self.beta * chosen_avg_log_prob
        implicit_reward_rejected = self.beta * rejected_avg_log_prob

        loss = -F.logsigmoid(implicit_reward_chosen - implicit_reward_rejected - self.gamma).mean()

        self.optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(self.policy.parameters(), self.max_grad_norm)
        self.optimizer.step()

        with torch.no_grad():
            accuracy = (implicit_reward_chosen > implicit_reward_rejected).float().mean()
            margin = (implicit_reward_chosen - implicit_reward_rejected).mean()

        metrics = {
            'loss': loss.item(),
            'accuracy': accuracy.item(),
            'margin': margin.item(),
            'chosen_reward': implicit_reward_chosen.mean().item(),
            'rejected_reward': implicit_reward_rejected.mean().item(),
        }
        self.metrics_history.append(metrics)
        return metrics

policy = SimplePolicy()
simpo_trainer = SimPOTrainer(policy, beta=0.1, gamma=0.5, lr=1e-3)

x = torch.randn(32, 64)
y_chosen = torch.randint(0, 50, (32,))
y_rejected = torch.randint(0, 50, (32,))
len_chosen = torch.randint(5, 20, (32,))
len_rejected = torch.randint(5, 20, (32,))

print('=== SimPO Training ===')
for step in range(30):
    metrics = simpo_trainer.train_step(x, y_chosen, y_rejected, len_chosen, len_rejected)
    if (step + 1) % 10 == 0:
        print(f'Step {step+1:3d}: loss={metrics["loss"]:.4f}, '
              f'acc={metrics["accuracy"]:.2%}, margin={metrics["margin"]:.4f}')

print(f'\nKey: SimPO needs NO reference model, saving 50% GPU memory.')
print(f'gamma ensures a minimum reward gap between chosen and rejected.')

## 5. 方法对比实验

在相同数据上对比四种对齐方法的训练表现。

In [ ]:
def compare_methods(n_steps=50):
    d, v = 64, 50
    x = torch.randn(32, d)
    y_w = torch.randint(0, v, (32,))
    y_l = torch.randint(0, v, (32,))
    labels = torch.randint(0, v, (32,))
    is_desirable = torch.tensor([1,1,0,1,0,0,1,0,1,1,0,1,1,0,1,0,
                                  1,0,1,1,0,1,0,0,1,1,0,1,0,1,1,0])

    results = {'DPO': [], 'KTO': [], 'ORPO': [], 'SimPO': []}

    policy_dpo = SimplePolicy(d, v)
    policy_kto = SimplePolicy(d, v)
    policy_orpo = SimplePolicy(d, v)
    policy_simpo = SimplePolicy(d, v)
    ref = SimplePolicy(d, v)
    ref.load_state_dict(policy_dpo.state_dict())
    for p in ref.parameters():
        p.requires_grad = False

    opt_dpo = torch.optim.AdamW(policy_dpo.parameters(), lr=1e-3)
    opt_kto = torch.optim.AdamW(policy_kto.parameters(), lr=1e-3)
    opt_orpo = torch.optim.AdamW(policy_orpo.parameters(), lr=1e-3)
    opt_simpo = torch.optim.AdamW(policy_simpo.parameters(), lr=1e-3)

    for step in range(n_steps):
        with torch.no_grad():
            ref_logits = ref(x)

        logits = policy_dpo(x)
        log_pi_w = F.log_softmax(logits, dim=-1).gather(1, y_w.unsqueeze(1)).squeeze(1)
        log_pi_l = F.log_softmax(logits, dim=-1).gather(1, y_l.unsqueeze(1)).squeeze(1)
        log_ref_w = F.log_softmax(ref_logits, dim=-1).gather(1, y_w.unsqueeze(1)).squeeze(1)
        log_ref_l = F.log_softmax(ref_logits, dim=-1).gather(1, y_l.unsqueeze(1)).squeeze(1)
        loss_dpo = -F.logsigmoid(0.1 * ((log_pi_w - log_ref_w) - (log_pi_l - log_ref_l))).mean()
        opt_dpo.zero_grad()
        loss_dpo.backward()
        opt_dpo.step()
        results['DPO'].append(loss_dpo.item())

        logits = policy_kto(x)
        log_pi = F.log_softmax(logits, dim=-1).gather(1, labels.unsqueeze(1)).squeeze(1)
        log_ref = F.log_softmax(ref_logits, dim=-1).gather(1, labels.unsqueeze(1)).squeeze(1)
        z = log_pi - log_ref
        loss_des = 1 - torch.sigmoid(0.1 * z)
        loss_undes = 1 - torch.sigmoid(0.1 * (-z))
        loss_kto = torch.where(is_desirable.bool(), loss_des, loss_undes).mean()
        opt_kto.zero_grad()
        loss_kto.backward()
        opt_kto.step()
        results['KTO'].append(loss_kto.item())

        logits = policy_orpo(x)
        probs = F.softmax(logits, dim=-1)
        p_w = probs.gather(1, y_w.unsqueeze(1)).squeeze(1)
        p_l = probs.gather(1, y_l.unsqueeze(1)).squeeze(1)
        sft_loss = -F.log_softmax(logits, dim=-1).gather(1, y_w.unsqueeze(1)).squeeze(1).mean()
        odds_ratio = (p_w / (1 - p_w + 1e-8)) / (p_l / (1 - p_l + 1e-8) + 1e-8)
        or_loss = -F.logsigmoid(torch.log(odds_ratio + 1e-8)).mean()
        loss_orpo = sft_loss + 0.1 * or_loss
        opt_orpo.zero_grad()
        loss_orpo.backward()
        opt_orpo.step()
        results['ORPO'].append(loss_orpo.item())

        logits = policy_simpo(x)
        log_probs = F.log_softmax(logits, dim=-1)
        r_w = 0.1 * log_probs.gather(1, y_w.unsqueeze(1)).squeeze(1)
        r_l = 0.1 * log_probs.gather(1, y_l.unsqueeze(1)).squeeze(1)
        loss_simpo = -F.logsigmoid(r_w - r_l - 0.5).mean()
        opt_simpo.zero_grad()
        loss_simpo.backward()
        opt_simpo.step()
        results['SimPO'].append(loss_simpo.item())

    return results

results = compare_methods(n_steps=50)

print('=== Alignment Methods Comparison ===')
for method, losses in results.items():
    print(f'{method:6s}: initial={losses[0]:.4f}, '
          f'final={losses[-1]:.4f}, '
          f'improvement={losses[0]-losses[-1]:.4f}')

print(f'\nKey: All methods converge. Choose based on data availability and constraints.')
print(f'DPO: paired data + reference model. KTO: unpaired data. ORPO/SimPO: no reference model.')

## 6. 对齐方法选择指南

### 根据数据类型选择
| 数据类型 | 推荐方法 | 原因 |
|----------|----------|------|
| 成对偏好数据 | DPO | 直接优化偏好对，效果最好 |
| 二元好/坏标签 | KTO | 不需要配对，数据获取简单 |
| 仅有SFT数据 | ORPO | 一步完成SFT+对齐 |
| 显存受限 | SimPO | 不需要参考模型 |

### 根据场景选择
| 场景 | 推荐方法 | 原因 |
|------|----------|------|
| 大规模对齐 | RLAIF + DPO | AI标注降低成本 |
| 快速原型 | SimPO | 最简单，无需参考模型 |
| 追求最优性能 | RLHF | 在线探索，上限最高 |
| 数据有噪声 | cDPO / KTO | 对噪声鲁棒 |

### 工业实践组合方案
1. **标准方案**：SFT → RLAIF生成偏好数据 → DPO对齐
2. **快速方案**：SFT → ORPO（一步完成）
3. **高性能方案**：SFT → DPO → RLHF精调
4. **低成本方案**：SFT → KTO（用二元标签）

### 关键注意事项
- **对齐税（Alignment Tax）**：对齐可能损害模型能力，需要在对齐和能力之间平衡
- **SFT数据混合**：对齐训练时混入10-20%的SFT数据，防止能力退化
- **迭代对齐**：多次迭代（对齐→评估→收集数据→再对齐）比一次对齐效果更好
- **多维度对齐**：有用性、安全性、诚实性需要分别优化，不能只优化一个维度